In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="ollama:llama3.1:8b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="ollama:llama3.1:8b",
            trigger=("messages", 6),
            keep=("messages", 1)
        )
    ],
)

In [10]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n* The user's primary goal is to gather information about the moon, specifically Lunapolis.\n\n## SUMMARY\n\n* The capital of the moon is Lunapolis.\n* The weather in Lunapolis is currently clear, with extreme temperature fluctuations (high of 120C and low of -100C).\n* There are 100,000 cheese miners living in Lunapolis.\n* The cheese miners' union is considering a strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\n* None\n\n## NEXT STEPS\n\n* Research the cheese miners' union and their demands to better understand the strike situation.\n* Investigate the new president and their policies to determine the root cause of the union's dissatisfaction.\n* Continue gathering information about Lunapolis and its inhabitants to improve knowledge and understanding of the moon's capital.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='10975c

In [11]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

* The user's primary goal is to gather information about the moon, specifically Lunapolis.

## SUMMARY

* The capital of the moon is Lunapolis.
* The weather in Lunapolis is currently clear, with extreme temperature fluctuations (high of 120C and low of -100C).
* There are 100,000 cheese miners living in Lunapolis.
* The cheese miners' union is considering a strike due to dissatisfaction with the new president.

## ARTIFACTS

* None

## NEXT STEPS

* Research the cheese miners' union and their demands to better understand the strike situation.
* Investigate the new president and their policies to determine the root cause of the union's dissatisfaction.
* Continue gathering information about Lunapolis and its inhabitants to improve knowledge and understanding of the moon's capital.


## Trim/delete messages

In [12]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [13]:
agent = create_agent(
    model="ollama:llama3.1:8b",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [14]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='f5108d44-b3ae-49df-bedf-347b9d7a9fac'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='934e4efb-71e9-4734-8e89-743213d97c3c', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='1325c3cd-bfc2-4983-b3bf-67ac2ce89af8'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='12595a5f-398f-4246-a454-0837dbf737bc', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='b43fbcf0-6650-4338-816b-955730b0880e'),
              AIMessage(content="A good question! If the device is overheating, it may not turn

In [ ]:
print(response["messages"][-1].content)